# Support Vector Machines: Complete Mathematical Guide & Implementation

## Learning Objectives
By the end of this notebook, you will understand:
- Complete mathematical foundation of Support Vector Machines (SVMs)
- Hard and soft margin optimization with Lagrangian dual formulation
- Kernel trick and various kernel functions with geometric intuition
- Sequential Minimal Optimization (SMO) algorithm implementation
- Multi-class extensions and advanced SVM techniques
- Performance comparison with other machine learning algorithms
- Real-world applications in text classification, image recognition, and bioinformatics

## Table of Contents
1. **Mathematical Foundation & Geometric Intuition**
2. **Hard Margin SVM & Optimization Theory**
3. **Soft Margin SVM & C Parameter**
4. **Kernel Trick & Non-linear Classification**
5. **Support Vector Analysis & Sparsity**
6. **Sequential Minimal Optimization (SMO)**
7. **Multi-class SVM Extensions**
8. **Implementation from Scratch**
9. **Real-World Applications & Performance Analysis**
10. **Advanced Topics & Future Directions**

---

## 1. Mathematical Foundation & Geometric Intuition

### What are Support Vector Machines?

Support Vector Machines (SVMs) are **maximum margin classifiers** that find the optimal hyperplane separating different classes. The key insight is that the best decision boundary is the one that maximizes the distance (margin) to the nearest data points from both classes.

### 1.1 Linear Separability & Hyperplanes

Given training data $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$ where:
- $\mathbf{x}_i \in \mathbb{R}^d$ are feature vectors
- $y_i \in \{-1, +1\}$ are class labels

A **hyperplane** in $d$-dimensional space is defined as:
$$\mathbf{w}^T\mathbf{x} + b = 0$$

Where:
- $\mathbf{w} \in \mathbb{R}^d$ is the **normal vector** (defines orientation)
- $b \in \mathbb{R}$ is the **bias term** (defines position)
- $\|\mathbf{w}\| = 1$ for normalized hyperplane

### 1.2 Decision Function & Classification

The **decision function** for a new point $\mathbf{x}$ is:
$$f(\mathbf{x}) = \text{sign}(\mathbf{w}^T\mathbf{x} + b)$$

The **signed distance** from point $\mathbf{x}_i$ to hyperplane is:
$$d_i = \frac{y_i(\mathbf{w}^T\mathbf{x}_i + b)}{\|\mathbf{w}\|}$$

### 1.3 Margin & Support Vectors

The **margin** $\gamma$ is the minimum distance from any training point to the decision boundary:
$$\gamma = \min_{i=1,\ldots,n} \frac{y_i(\mathbf{w}^T\mathbf{x}_i + b)}{\|\mathbf{w}\|}$$

**Support Vectors** are the training points that lie exactly on the margin boundary:
- These are the points with minimum distance to the hyperplane
- Removing non-support vectors doesn't change the optimal hyperplane
- Support vectors determine the entire solution (**sparsity property**)

---

## 2. Hard Margin SVM & Optimization Theory

### 2.1 Hard Margin Formulation

For **linearly separable data**, we want to find the hyperplane that maximizes the margin. 

**Canonical form**: Scale $\mathbf{w}$ and $b$ such that the closest points satisfy:
$$\min_{i} y_i(\mathbf{w}^T\mathbf{x}_i + b) = 1$$

This gives us the constraint:
$$y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 \quad \forall i = 1, \ldots, n$$

The margin becomes: $\gamma = \frac{1}{\|\mathbf{w}\|}$

### 2.2 Primal Optimization Problem

**Maximize margin** $\Leftrightarrow$ **Minimize** $\|\mathbf{w}\|^2$:

$$\begin{align}
\min_{\mathbf{w}, b} \quad &\frac{1}{2}\|\mathbf{w}\|^2 \\
\text{subject to} \quad &y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1, \quad i = 1, \ldots, n
\end{align}$$

This is a **convex quadratic programming** problem with linear constraints.

### 2.3 Lagrangian Dual Formulation

Introduce Lagrange multipliers $\alpha_i \geq 0$:

$$L(\mathbf{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}\|\mathbf{w}\|^2 - \sum_{i=1}^n \alpha_i[y_i(\mathbf{w}^T\mathbf{x}_i + b) - 1]$$

**KKT Conditions**:
1. $\nabla_{\mathbf{w}} L = \mathbf{w} - \sum_{i=1}^n \alpha_i y_i \mathbf{x}_i = 0 \Rightarrow \mathbf{w} = \sum_{i=1}^n \alpha_i y_i \mathbf{x}_i$
2. $\frac{\partial L}{\partial b} = -\sum_{i=1}^n \alpha_i y_i = 0 \Rightarrow \sum_{i=1}^n \alpha_i y_i = 0$
3. $\alpha_i \geq 0$ and $\alpha_i[y_i(\mathbf{w}^T\mathbf{x}_i + b) - 1] = 0$ (complementary slackness)

### 2.4 Dual Problem

Substituting KKT conditions into the Lagrangian:

$$\begin{align}
\max_{\boldsymbol{\alpha}} \quad &\sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i=1}^n\sum_{j=1}^n \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T\mathbf{x}_j \\
\text{subject to} \quad &\sum_{i=1}^n \alpha_i y_i = 0 \\
&\alpha_i \geq 0, \quad i = 1, \ldots, n
\end{align}$$

### 2.5 Support Vector Identification

From complementary slackness:
- $\alpha_i = 0 \Rightarrow$ point $\mathbf{x}_i$ is not a support vector
- $\alpha_i > 0 \Rightarrow$ point $\mathbf{x}_i$ is a support vector (lies on margin boundary)

**Decision function**:
$$f(\mathbf{x}) = \text{sign}\left(\sum_{i=1}^n \alpha_i y_i \mathbf{x}_i^T\mathbf{x} + b\right)$$

Only support vectors (with $\alpha_i > 0$) contribute to the sum!

---

## 3. Soft Margin SVM & C Parameter

### 3.1 Handling Non-separable Data

Real-world data is often **not linearly separable** due to:
- **Outliers**: Mislabeled or noisy data points
- **Overlapping classes**: Inherent class overlap in feature space

**Solution**: Allow some misclassification while still maximizing margin.

### 3.2 Slack Variables

Introduce **slack variables** $\xi_i \geq 0$:
- $\xi_i = 0$: Point is correctly classified and outside margin
- $0 < \xi_i < 1$: Point is correctly classified but inside margin
- $\xi_i = 1$: Point is on the decision boundary
- $\xi_i > 1$: Point is misclassified

### 3.3 Soft Margin Optimization

$$\begin{align}
\min_{\mathbf{w}, b, \boldsymbol{\xi}} \quad &\frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^n \xi_i \\
\text{subject to} \quad &y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 - \xi_i, \quad i = 1, \ldots, n \\
&\xi_i \geq 0, \quad i = 1, \ldots, n
\end{align}$$

**Parameter C**:
- **Large C**: Hard margin (low bias, high variance)
- **Small C**: Soft margin (high bias, low variance)
- Controls **bias-variance tradeoff**

### 3.4 Soft Margin Dual Problem

$$\begin{align}
\max_{\boldsymbol{\alpha}} \quad &\sum_{i=1}^n \alpha_i - \frac{1}{2}\sum_{i=1}^n\sum_{j=1}^n \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T\mathbf{x}_j \\
\text{subject to} \quad &\sum_{i=1}^n \alpha_i y_i = 0 \\
&0 \leq \alpha_i \leq C, \quad i = 1, \ldots, n
\end{align}$$

**Support Vector Categories**:
- $\alpha_i = 0$: Non-support vector (correctly classified, outside margin)
- $0 < \alpha_i < C$: Support vector on margin boundary ($\xi_i = 0$)
- $\alpha_i = C$: Support vector inside margin or misclassified ($\xi_i > 0$)

---

## 4. Kernel Trick & Non-linear Classification

### 4.1 Limitation of Linear SVMs

Linear SVMs can only find **linear decision boundaries**. For complex, non-linear patterns, we need to map data to higher-dimensional space.

### 4.2 Feature Mapping

Map input space $\mathbb{R}^d$ to higher-dimensional feature space $\mathcal{H}$:
$$\phi: \mathbb{R}^d \rightarrow \mathcal{H}$$

Example: For $\mathbf{x} = (x_1, x_2)$, quadratic mapping:
$$\phi(\mathbf{x}) = (x_1^2, \sqrt{2}x_1x_2, x_2^2, \sqrt{2}x_1, \sqrt{2}x_2, 1)$$

**Problem**: Feature space can be very high-dimensional (even infinite!)

### 4.3 The Kernel Trick

**Key insight**: SVM dual only requires **inner products** $\mathbf{x}_i^T\mathbf{x}_j$.

Define **kernel function**:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^T\phi(\mathbf{x}_j)$$

**Kernel trick**: Compute $K(\mathbf{x}_i, \mathbf{x}_j)$ directly without explicit mapping $\phi(\cdot)$!

### 4.4 Popular Kernel Functions

#### **Linear Kernel**
$$K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i^T\mathbf{x}_j$$
- Equivalent to no kernel (original space)
- Best for linearly separable, high-dimensional data

#### **Polynomial Kernel**
$$K(\mathbf{x}_i, \mathbf{x}_j) = (\mathbf{x}_i^T\mathbf{x}_j + c)^d$$
- **d**: degree of polynomial
- **c**: trade-off between high/low degree terms
- Captures interactions between features

#### **Radial Basis Function (RBF/Gaussian) Kernel**
$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$
- **γ**: controls width of RBF (inverse of variance)
- **High γ**: Narrow RBF, complex boundaries (potential overfitting)
- **Low γ**: Wide RBF, smooth boundaries (potential underfitting)
- Most popular for non-linear classification

#### **Sigmoid Kernel**
$$K(\mathbf{x}_i, \mathbf{x}_j) = \tanh(\gamma\mathbf{x}_i^T\mathbf{x}_j + c)$$
- Behaves like neural network
- Not always positive definite (may not be valid kernel)

### 4.5 Mercer's Condition

For a function to be a valid kernel, it must satisfy **Mercer's condition**:
$$\iint K(\mathbf{x}, \mathbf{z})g(\mathbf{x})g(\mathbf{z})d\mathbf{x}d\mathbf{z} \geq 0$$

for all square-integrable functions $g$.

**Practical implication**: Kernel matrix must be positive semi-definite.

---

## 5. Sequential Minimal Optimization (SMO)

### 5.1 Challenges of Quadratic Programming

Standard QP solvers have **cubic time complexity** $O(n^3)$, which is prohibitive for large datasets.

**SMO insight**: Optimize only **two variables** at a time while keeping all others fixed.

### 5.2 SMO Algorithm Overview

1. **Select two variables** $\alpha_i, \alpha_j$ to optimize
2. **Optimize analytically** with respect to these two variables
3. **Update** $\alpha_i, \alpha_j$ and bias $b$
4. **Repeat** until convergence

### 5.3 Variable Selection Heuristics

**First variable**: Choose $\alpha_i$ that violates KKT conditions most:
- $\alpha_i = 0$ but $y_if(\mathbf{x}_i) < 1$ (should increase $\alpha_i$)
- $\alpha_i = C$ but $y_if(\mathbf{x}_i) > 1$ (should decrease $\alpha_i$)
- $0 < \alpha_i < C$ but $y_if(\mathbf{x}_i) \neq 1$ (should be exactly 1)

**Second variable**: Choose $\alpha_j$ that maximizes step size:
$$|E_i - E_j|$$
where $E_i = f(\mathbf{x}_i) - y_i$ is the prediction error.

### 5.4 Analytical Solution

For fixed all other $\alpha_k$ (k ≠ i,j), minimize:
$$W(\alpha_i, \alpha_j) = \frac{1}{2}K_{ii}\alpha_i^2 + \frac{1}{2}K_{jj}\alpha_j^2 + yiy_jK_{ij}\alpha_i\alpha_j + y_i\alpha_i v_i + y_j\alpha_j v_j$$

Subject to:
- $y_i\alpha_i + y_j\alpha_j = \gamma$ (constant)
- $0 \leq \alpha_i, \alpha_j \leq C$

**Optimal update**:
$$\alpha_j^{new} = \alpha_j^{old} + \frac{y_j(E_i - E_j)}{\eta}$$

where $\eta = K_{ii} + K_{jj} - 2K_{ij}$

### 5.5 Convergence Properties

SMO has **linear convergence** and typically converges much faster than standard QP solvers for large, sparse problems.

---

## 6. Multi-class SVM Extensions

### 6.1 Binary vs Multi-class

SVMs are inherently **binary classifiers**. For multi-class problems with $K > 2$ classes, we need extensions:

### 6.2 One-vs-Rest (OvR)

Train $K$ binary classifiers:
- Classifier $k$: Class $k$ vs all other classes
- **Prediction**: Class with highest decision function value

**Advantages**: Simple, efficient
**Disadvantages**: Unbalanced training sets, inconsistent regions

### 6.3 One-vs-One (OvO)

Train $\binom{K}{2}$ binary classifiers:
- For each pair of classes $(i,j)$: Class $i$ vs Class $j$
- **Prediction**: Majority voting

**Advantages**: Balanced training sets, better performance
**Disadvantages**: More classifiers to train

### 6.4 Error-Correcting Output Codes (ECOC)

Encode each class with a **binary codeword**:
- Train one binary classifier per bit position
- **Prediction**: Class with minimum Hamming distance

**Advantages**: Error correction capability, theoretically motivated
**Disadvantages**: Complex design, computational overhead

---

## 7. Advanced Topics & Applications

### 7.1 SVM Regression (SVR)

Extend SVM to regression by using **ε-insensitive loss**:
$$L_ε(y, f(x)) = \max(0, |y - f(x)| - ε)$$

### 7.2 Online SVMs

Handle streaming data with **incremental learning**:
- Add new samples without retraining from scratch
- Remove samples (decremental learning)

### 7.3 Structured SVMs

Extend to **structured prediction**:
- Sequence labeling, parsing, image segmentation
- Joint feature mapping over input-output pairs

### 7.4 Real-World Applications

- **Text Classification**: Document categorization, spam filtering
- **Image Recognition**: Object detection, face recognition
- **Bioinformatics**: Protein classification, gene expression analysis
- **Finance**: Credit scoring, algorithmic trading
- **Medical Diagnosis**: Disease classification from symptoms/images

---

## 8. Advantages and Limitations

### 8.1 Advantages
- **Effective in high dimensions**: Works well when #features > #samples
- **Memory efficient**: Only stores support vectors
- **Versatile**: Different kernels for different problems
- **Theoretically well-founded**: Based on statistical learning theory

### 8.2 Limitations
- **No probabilistic output**: Only decision function (can be addressed with Platt scaling)
- **Sensitive to feature scaling**: Requires normalization
- **Kernel/hyperparameter selection**: Requires cross-validation
- **Large datasets**: Quadratic complexity in #samples for training

---

In [ ]:
# Essential Libraries and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, make_moons, load_digits, fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, SVR
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                           precision_recall_curve, roc_curve, roc_auc_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import warnings
import time
from scipy import stats
from itertools import combinations

warnings.filterwarnings('ignore')

# Configuration for better visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
sns.set_palette("husl")

print("🔰 Support Vector Machines: Complete Implementation & Analysis")
print("=" * 65)
print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn available for comparison and benchmarking")
print("=" * 65)

In [ ]:
# Complete SVM Implementation from Scratch

class KernelFunctions:
    """Collection of kernel functions for SVM"""
    
    @staticmethod
    def linear(X1, X2):
        """Linear kernel: K(x, z) = x^T z"""
        return np.dot(X1, X2.T)
    
    @staticmethod
    def polynomial(X1, X2, degree=3, coef0=1):
        """Polynomial kernel: K(x, z) = (x^T z + coef0)^degree"""
        return (np.dot(X1, X2.T) + coef0) ** degree
    
    @staticmethod
    def rbf(X1, X2, gamma=1.0):
        """RBF/Gaussian kernel: K(x, z) = exp(-gamma ||x - z||^2)"""
        if X1.ndim == 1:
            X1 = X1.reshape(1, -1)
        if X2.ndim == 1:
            X2 = X2.reshape(1, -1)
            
        # Compute squared Euclidean distances using broadcasting
        sq_dists = np.sum(X1**2, axis=1).reshape(-1, 1) + \
                  np.sum(X2**2, axis=1) - 2*np.dot(X1, X2.T)
        return np.exp(-gamma * sq_dists)
    
    @staticmethod
    def sigmoid(X1, X2, gamma=1.0, coef0=0):
        """Sigmoid kernel: K(x, z) = tanh(gamma x^T z + coef0)"""
        return np.tanh(gamma * np.dot(X1, X2.T) + coef0)


class SoftMarginSVM:
    """
    Soft Margin SVM with SMO algorithm implementation
    Supports multiple kernels and handles non-separable data
    """
    
    def __init__(self, C=1.0, kernel='rbf', gamma='scale', degree=3, 
                 coef0=0, tol=1e-3, max_iter=1000):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.tol = tol
        self.max_iter = max_iter
        
        # Initialize parameters
        self.alpha = None
        self.b = 0
        self.X_train = None
        self.y_train = None
        self.support_vectors_ = None
        self.support_vector_labels_ = None
        self.n_support_vectors_ = None
        
    def _get_kernel_function(self):
        """Get the appropriate kernel function"""
        if self.kernel == 'linear':
            return KernelFunctions.linear
        elif self.kernel == 'poly':
            return lambda X1, X2: KernelFunctions.polynomial(
                X1, X2, self.degree, self.coef0)
        elif self.kernel == 'rbf':
            gamma = self._compute_gamma()
            return lambda X1, X2: KernelFunctions.rbf(X1, X2, gamma)
        elif self.kernel == 'sigmoid':
            gamma = self._compute_gamma()
            return lambda X1, X2: KernelFunctions.sigmoid(
                X1, X2, gamma, self.coef0)
        else:
            raise ValueError(f"Unknown kernel: {self.kernel}")
    
    def _compute_gamma(self):
        """Compute gamma parameter for RBF and sigmoid kernels"""
        if self.gamma == 'scale':
            return 1.0 / (self.X_train.shape[1] * np.var(self.X_train))
        elif self.gamma == 'auto':
            return 1.0 / self.X_train.shape[1]
        else:
            return self.gamma
    
    def _kernel_matrix(self, X1, X2=None):
        """Compute kernel matrix between X1 and X2 (or X1 and itself)"""
        if X2 is None:
            X2 = X1
        
        kernel_func = self._get_kernel_function()
        return kernel_func(X1, X2)
    
    def _compute_error(self, i):
        """Compute prediction error for sample i"""
        decision_value = np.sum(self.alpha * self.y_train * 
                               self.K_train[:, i]) + self.b
        return decision_value - self.y_train[i]
    
    def _select_second_alpha(self, i, E_i):
        """Select second alpha using maximum step size heuristic"""
        valid_alphas = np.where((self.alpha > 0) & (self.alpha < self.C))[0]
        
        if len(valid_alphas) > 1:
            errors = np.array([self._compute_error(k) for k in valid_alphas])
            # Choose j that maximizes |E_i - E_j|
            j_idx = np.argmax(np.abs(errors - E_i))
            return valid_alphas[j_idx]
        else:
            # Random selection
            candidates = list(range(len(self.alpha)))
            candidates.remove(i)
            return np.random.choice(candidates)
    
    def _clip_alpha(self, alpha, L, H):
        """Clip alpha to [L, H] interval"""
        if alpha > H:
            return H
        elif alpha < L:
            return L
        else:
            return alpha
    
    def _update_bias(self, i, j, alpha_i_old, alpha_j_old, E_i, E_j):
        """Update bias term"""
        b1 = self.b - E_i - self.y_train[i] * (self.alpha[i] - alpha_i_old) * self.K_train[i, i] - \
             self.y_train[j] * (self.alpha[j] - alpha_j_old) * self.K_train[i, j]
        
        b2 = self.b - E_j - self.y_train[i] * (self.alpha[i] - alpha_i_old) * self.K_train[i, j] - \
             self.y_train[j] * (self.alpha[j] - alpha_j_old) * self.K_train[j, j]
        
        if 0 < self.alpha[i] < self.C:
            self.b = b1
        elif 0 < self.alpha[j] < self.C:
            self.b = b2
        else:
            self.b = (b1 + b2) / 2
    
    def _violates_kkt(self, i):
        """Check if sample i violates KKT conditions"""
        r_i = self.y_train[i] * self._compute_error(i)
        
        if (self.alpha[i] < self.C and r_i < -self.tol):
            return True
        elif (self.alpha[i] > 0 and r_i > self.tol):
            return True
        return False
    
    def _smo_step(self, i, j):
        """Perform one step of SMO optimization"""
        if i == j:
            return False
            
        alpha_i_old = self.alpha[i]
        alpha_j_old = self.alpha[j]
        
        # Compute errors
        E_i = self._compute_error(i)
        E_j = self._compute_error(j)
        
        # Compute bounds L and H
        if self.y_train[i] != self.y_train[j]:
            L = max(0, self.alpha[j] - self.alpha[i])
            H = min(self.C, self.C + self.alpha[j] - self.alpha[i])
        else:
            L = max(0, self.alpha[i] + self.alpha[j] - self.C)
            H = min(self.C, self.alpha[i] + self.alpha[j])
            
        if L == H:
            return False
            
        # Compute eta (second derivative of objective function)
        eta = self.K_train[i, i] + self.K_train[j, j] - 2 * self.K_train[i, j]
        
        if eta <= 0:
            return False
            
        # Update alpha_j
        self.alpha[j] += self.y_train[j] * (E_i - E_j) / eta
        self.alpha[j] = self._clip_alpha(self.alpha[j], L, H)
        
        # Check for significant change
        if abs(self.alpha[j] - alpha_j_old) < 1e-5:
            return False
            
        # Update alpha_i
        self.alpha[i] += self.y_train[i] * self.y_train[j] * (alpha_j_old - self.alpha[j])
        
        # Update bias
        self._update_bias(i, j, alpha_i_old, alpha_j_old, E_i, E_j)
        
        return True
    
    def fit(self, X, y):
        """Train the SVM using SMO algorithm"""
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        
        # Convert labels to {-1, +1}
        if set(np.unique(y)) == {0, 1}:
            self.y_train = np.where(y == 0, -1, 1)
        
        n_samples = len(X)
        
        # Initialize alphas and bias
        self.alpha = np.zeros(n_samples)
        self.b = 0
        
        # Compute kernel matrix (cached for efficiency)
        self.K_train = self._kernel_matrix(self.X_train)
        
        # SMO main loop
        num_changed = 0
        examine_all = True
        iteration = 0
        
        while (num_changed > 0 or examine_all) and iteration < self.max_iter:
            num_changed = 0
            
            if examine_all:
                # Examine all training examples
                for i in range(n_samples):
                    if self._violates_kkt(i):
                        j = self._select_second_alpha(i, self._compute_error(i))
                        if self._smo_step(i, j):
                            num_changed += 1
            else:
                # Examine non-bound examples (0 < alpha < C)
                non_bound_indices = np.where((self.alpha > 0) & (self.alpha < self.C))[0]
                for i in non_bound_indices:
                    if self._violates_kkt(i):
                        j = self._select_second_alpha(i, self._compute_error(i))
                        if self._smo_step(i, j):
                            num_changed += 1
            
            if examine_all:
                examine_all = False
            elif num_changed == 0:
                examine_all = True
                
            iteration += 1
        
        # Extract support vectors
        support_mask = self.alpha > 1e-8  # Small threshold for numerical stability
        self.support_vectors_ = self.X_train[support_mask]
        self.support_vector_labels_ = self.y_train[support_mask]
        self.support_alphas_ = self.alpha[support_mask]
        self.n_support_vectors_ = len(self.support_vectors_)
        
        return self
    
    def decision_function(self, X):
        """Compute decision function values"""
        X = np.array(X)
        K = self._kernel_matrix(self.support_vectors_, X)
        
        decision_values = np.sum(
            (self.support_alphas_ * self.support_vector_labels_).reshape(-1, 1) * K,
            axis=0
        ) + self.b
        
        return decision_values
    
    def predict(self, X):
        """Make predictions"""
        return np.sign(self.decision_function(X))
    
    def predict_proba(self, X):
        """Estimate class probabilities using Platt scaling"""
        decision_values = self.decision_function(X)
        
        # Simple sigmoid mapping (not true Platt scaling)
        proba_pos = 1 / (1 + np.exp(-decision_values))
        proba_neg = 1 - proba_pos
        
        return np.column_stack([proba_neg, proba_pos])


# Utility functions for evaluation and visualization
def evaluate_svm(y_true, y_pred, model_name):
    """Comprehensive SVM evaluation"""
    accuracy = accuracy_score(y_true, y_pred)
    
    print(f"\\n📊 {model_name} Performance:")
    print(f"   • Accuracy:  {accuracy:.4f}")
    
    if hasattr(y_pred, 'support_vectors_'):
        print(f"   • Support Vectors: {len(y_pred.support_vectors_)}")
    
    return {'accuracy': accuracy}


def plot_svm_decision_boundary(X, y, model, title="SVM Decision Boundary"):
    """Plot SVM decision boundary with support vectors"""
    plt.figure(figsize=(10, 8))
    
    # Create a mesh for plotting
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Get decision function values
    mesh_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.decision_function(mesh_points)
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary and margins
    plt.contour(xx, yy, Z, levels=[-1, 0, 1], alpha=0.5,
               linestyles=['--', '-', '--'], 
               colors=['red', 'black', 'red'],
               linewidths=[2, 3, 2])
    
    # Fill decision regions
    plt.contourf(xx, yy, Z, levels=50, alpha=0.3, cmap='RdYlBu')
    
    # Plot data points
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', 
                         s=50, edgecolors='black', alpha=0.8)
    
    # Highlight support vectors
    if hasattr(model, 'support_vectors_'):
        plt.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                   s=200, facecolors='none', edgecolors='green', 
                   linewidth=3, label=f'Support Vectors ({len(model.support_vectors_)})')
        plt.legend()
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.colorbar(scatter, label='Class')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    return plt.gcf()

print("\\n✅ Complete SVM implementation ready!")
print("   • SoftMarginSVM: Custom implementation with SMO algorithm")
print("   • KernelFunctions: Linear, Polynomial, RBF, and Sigmoid kernels")
print("   • Visualization tools: Decision boundary and support vector plotting")
print("   • Evaluation utilities: Comprehensive performance metrics")

In [ ]:
# 🔮 COMPREHENSIVE KERNEL ANALYSIS AND VISUALIZATION
print("🔮 COMPREHENSIVE KERNEL ANALYSIS AND VISUALIZATION")
print("=" * 60)

# Create different types of datasets to demonstrate kernel capabilities
def create_benchmark_datasets():
    """Create various datasets to test different kernel performance"""
    datasets = {}
    
    # 1. Linear separable data
    X1, y1 = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                                n_informative=2, n_clusters_per_class=1, 
                                class_sep=2.0, random_state=42)
    datasets['Linear Separable'] = (X1, y1)
    
    # 2. Non-linear circular pattern
    X2, y2 = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)
    datasets['Circular Pattern'] = (X2, y2)
    
    # 3. Non-linear moon pattern
    X3, y3 = make_moons(n_samples=200, noise=0.15, random_state=42)
    datasets['Moon Pattern'] = (X3, y3)
    
    # 4. Overlapping classes
    X4, y4 = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                n_informative=2, n_clusters_per_class=2,
                                class_sep=0.8, random_state=42)
    datasets['Overlapping Classes'] = (X4, y4)
    
    # 5. High-dimensional projection (XOR-like)
    np.random.seed(42)
    n_samples = 200
    X5 = np.random.randn(n_samples, 2)
    y5 = ((X5[:, 0] * X5[:, 1]) > 0).astype(int)
    datasets['XOR Pattern'] = (X5, y5)
    
    return datasets

datasets = create_benchmark_datasets()

# Define kernels to compare
kernels_config = {
    'Linear': {'kernel': 'linear', 'C': 1.0},
    'Polynomial (d=2)': {'kernel': 'poly', 'degree': 2, 'C': 1.0, 'coef0': 1},
    'Polynomial (d=3)': {'kernel': 'poly', 'degree': 3, 'C': 1.0, 'coef0': 1},
    'RBF (γ=1.0)': {'kernel': 'rbf', 'C': 1.0, 'gamma': 1.0},
    'RBF (γ=0.1)': {'kernel': 'rbf', 'C': 1.0, 'gamma': 0.1},
    'Sigmoid': {'kernel': 'sigmoid', 'C': 1.0, 'gamma': 1.0, 'coef0': 0}
}

# Function to plot decision boundaries for multiple kernels
def plot_kernel_comparison(X, y, kernels_config, dataset_name):
    """Plot decision boundaries for different kernels on the same dataset"""
    n_kernels = len(kernels_config)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.ravel()
    
    # Standardize data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42)
    
    results = {}
    
    for i, (kernel_name, params) in enumerate(kernels_config.items()):
        ax = axes[i]
        
        # Train SVM with scikit-learn for reliable results
        svm = SVC(**params)
        svm.fit(X_train, y_train)
        
        # Make predictions
        y_pred = svm.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        # Store results
        results[kernel_name] = {
            'accuracy': accuracy,
            'n_support_vectors': len(svm.support_vectors_),
            'support_ratio': len(svm.support_vectors_) / len(X_train)
        }
        
        # Plot decision boundary
        h = 0.02
        x_min, x_max = X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1
        y_min, y_max = X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                           np.arange(y_min, y_max, h))
        
        Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        # Plot decision boundary and margins
        ax.contour(xx, yy, Z, levels=[-1, 0, 1], alpha=0.5,
                  linestyles=['--', '-', '--'], 
                  colors=['red', 'black', 'red'], linewidths=[1, 2, 1])
        ax.contourf(xx, yy, Z, levels=50, alpha=0.3, cmap='RdYlBu')
        
        # Plot data points
        scatter = ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='RdYlBu', 
                           s=30, edgecolors='black', alpha=0.8)
        
        # Highlight support vectors
        ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
                  s=100, facecolors='none', edgecolors='green', linewidth=2)
        
        ax.set_title(f'{kernel_name}\nAcc: {accuracy:.3f}, SVs: {len(svm.support_vectors_)}',
                    fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('Feature 1')
        ax.set_ylabel('Feature 2')
    
    # Remove unused subplot if any
    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.suptitle(f'Kernel Comparison on {dataset_name} Dataset', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return results

# Compare kernels across all datasets
print("🎯 KERNEL PERFORMANCE ACROSS DIFFERENT DATA PATTERNS")
print("-" * 60)

all_results = {}
for dataset_name, (X, y) in datasets.items():
    print(f"\n📊 Dataset: {dataset_name}")
    results = plot_kernel_comparison(X, y, kernels_config, dataset_name)
    all_results[dataset_name] = results
    
    # Show best kernel for this dataset
    best_kernel = max(results.items(), key=lambda x: x[1]['accuracy'])
    print(f"   Best Kernel: {best_kernel[0]} (Accuracy: {best_kernel[1]['accuracy']:.3f})")

# Create summary performance table
print("\n📈 COMPREHENSIVE PERFORMANCE SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame()
for dataset_name, dataset_results in all_results.items():
    for kernel_name, metrics in dataset_results.items():
        summary_df = pd.concat([summary_df, pd.DataFrame({
            'Dataset': [dataset_name],
            'Kernel': [kernel_name],
            'Accuracy': [f"{metrics['accuracy']:.3f}"],
            'Support Vectors': [metrics['n_support_vectors']],
            'Support Ratio': [f"{metrics['support_ratio']:.3f}"]
        })], ignore_index=True)

# Display summary table
print(summary_df.to_string(index=False))

print("\n💡 KEY INSIGHTS FROM KERNEL ANALYSIS:")
print("=" * 45)
print("• Linear kernel works best for linearly separable data")
print("• RBF kernel is most versatile for non-linear patterns")  
print("• Polynomial kernels capture specific polynomial relationships")
print("• Higher γ in RBF creates more complex decision boundaries")
print("• Support vector ratio indicates model complexity")
print("• No single kernel dominates across all data types")

In [ ]:
# 📊 HYPERPARAMETER OPTIMIZATION AND VALIDATION CURVES
print("\n📊 HYPERPARAMETER OPTIMIZATION AND VALIDATION CURVES")
print("=" * 70)

# Use circular dataset for comprehensive hyperparameter analysis
X_circles, y_circles = make_circles(n_samples=400, noise=0.1, factor=0.3, random_state=42)
scaler = StandardScaler()
X_circles_scaled = scaler.fit_transform(X_circles)
X_train, X_test, y_train, y_test = train_test_split(
    X_circles_scaled, y_circles, test_size=0.3, random_state=42)

# 1. Effect of C parameter (regularization strength)
print("🔧 Analyzing the effect of C parameter (regularization strength)")
C_range = np.logspace(-3, 3, 20)

# Compute validation curves for C parameter
train_scores_C, val_scores_C = validation_curve(
    SVC(kernel='rbf', gamma='scale'), X_train, y_train,
    param_name='C', param_range=C_range,
    cv=5, scoring='accuracy', n_jobs=-1)

# 2. Effect of gamma parameter (RBF kernel width)
print("🔧 Analyzing the effect of gamma parameter (RBF kernel width)")
gamma_range = np.logspace(-4, 1, 20)

train_scores_gamma, val_scores_gamma = validation_curve(
    SVC(kernel='rbf', C=1.0), X_train, y_train,
    param_name='gamma', param_range=gamma_range,
    cv=5, scoring='accuracy', n_jobs=-1)

# 3. Grid search for optimal parameters
print("🔍 Performing comprehensive grid search...")
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1.0],
    'kernel': ['linear', 'rbf', 'poly']
}

grid_search = GridSearchCV(
    SVC(), param_grid, cv=5, scoring='accuracy', 
    n_jobs=-1, return_train_score=True)
grid_search.fit(X_train, y_train)

print(f"✅ Best parameters: {grid_search.best_params_}")
print(f"✅ Best CV score: {grid_search.best_score_:.4f}")
print(f"✅ Test accuracy: {grid_search.score(X_test, y_test):.4f}")

# Visualization
fig = plt.figure(figsize=(20, 12))

# Plot 1: C parameter validation curve
ax1 = plt.subplot(2, 4, 1)
train_mean_C = np.mean(train_scores_C, axis=1)
train_std_C = np.std(train_scores_C, axis=1)
val_mean_C = np.mean(val_scores_C, axis=1)
val_std_C = np.std(val_scores_C, axis=1)

ax1.semilogx(C_range, train_mean_C, 'o-', color='blue', label='Training score')
ax1.fill_between(C_range, train_mean_C - train_std_C, train_mean_C + train_std_C, alpha=0.1, color='blue')
ax1.semilogx(C_range, val_mean_C, 's-', color='red', label='Validation score')
ax1.fill_between(C_range, val_mean_C - val_std_C, val_mean_C + val_std_C, alpha=0.1, color='red')
ax1.set_xlabel('C (Regularization)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Validation Curve: C Parameter')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Gamma parameter validation curve
ax2 = plt.subplot(2, 4, 2)
train_mean_gamma = np.mean(train_scores_gamma, axis=1)
train_std_gamma = np.std(train_scores_gamma, axis=1)
val_mean_gamma = np.mean(val_scores_gamma, axis=1)
val_std_gamma = np.std(val_scores_gamma, axis=1)

ax2.semilogx(gamma_range, train_mean_gamma, 'o-', color='blue', label='Training score')
ax2.fill_between(gamma_range, train_mean_gamma - train_std_gamma, train_mean_gamma + train_std_gamma, alpha=0.1, color='blue')
ax2.semilogx(gamma_range, val_mean_gamma, 's-', color='red', label='Validation score')
ax2.fill_between(gamma_range, val_mean_gamma - val_std_gamma, val_mean_gamma + val_std_gamma, alpha=0.1, color='red')
ax2.set_xlabel('Gamma')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Curve: Gamma Parameter')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Heatmap of C vs Gamma performance
ax3 = plt.subplot(2, 4, 3)
C_vals = [0.1, 1, 10, 100]
gamma_vals = [0.001, 0.01, 0.1, 1.0]

# Create performance matrix
perf_matrix = np.zeros((len(C_vals), len(gamma_vals)))
for i, C in enumerate(C_vals):
    for j, gamma in enumerate(gamma_vals):
        svm_temp = SVC(kernel='rbf', C=C, gamma=gamma)
        svm_temp.fit(X_train, y_train)
        perf_matrix[i, j] = svm_temp.score(X_test, y_test)

im = ax3.imshow(perf_matrix, cmap='viridis', aspect='auto')
ax3.set_xticks(range(len(gamma_vals)))
ax3.set_xticklabels([f'{g:.3f}' for g in gamma_vals])
ax3.set_yticks(range(len(C_vals)))
ax3.set_yticklabels(C_vals)
ax3.set_xlabel('Gamma')
ax3.set_ylabel('C')
ax3.set_title('C vs Gamma Heatmap')
plt.colorbar(im, ax=ax3, label='Test Accuracy')

# Add text annotations
for i in range(len(C_vals)):
    for j in range(len(gamma_vals)):
        ax3.text(j, i, f'{perf_matrix[i, j]:.3f}', 
                ha='center', va='center', color='white', fontweight='bold')

# Plot 4: Best model decision boundary
ax4 = plt.subplot(2, 4, 4)
best_model = grid_search.best_estimator_

h = 0.02
x_min, x_max = X_circles_scaled[:, 0].min() - 1, X_circles_scaled[:, 0].max() + 1
y_min, y_max = X_circles_scaled[:, 1].min() - 1, X_circles_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = best_model.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax4.contour(xx, yy, Z, levels=[-1, 0, 1], alpha=0.5,
           linestyles=['--', '-', '--'], colors=['red', 'black', 'red'])
ax4.contourf(xx, yy, Z, levels=50, alpha=0.3, cmap='RdYlBu')
ax4.scatter(X_circles_scaled[:, 0], X_circles_scaled[:, 1], c=y_circles, 
           cmap='RdYlBu', s=30, edgecolors='black', alpha=0.8)
ax4.scatter(best_model.support_vectors_[:, 0], best_model.support_vectors_[:, 1],
           s=100, facecolors='none', edgecolors='green', linewidth=2)
ax4.set_title(f'Optimal SVM\\n{grid_search.best_params_}')
ax4.grid(True, alpha=0.3)

# Plot 5-8: Support vector analysis for different C values
C_demo_values = [0.1, 1, 10, 100]
for idx, C_val in enumerate(C_demo_values):
    ax = plt.subplot(2, 4, 5 + idx)
    
    svm_demo = SVC(kernel='rbf', C=C_val, gamma=1.0)
    svm_demo.fit(X_train, y_train)
    
    # Plot decision boundary
    Z = svm_demo.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], alpha=0.5,
              linestyles=['--', '-', '--'], colors=['red', 'black', 'red'])
    ax.contourf(xx, yy, Z, levels=50, alpha=0.3, cmap='RdYlBu')
    
    # Plot data points
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdYlBu', 
              s=30, edgecolors='black', alpha=0.8)
    
    # Highlight support vectors
    ax.scatter(svm_demo.support_vectors_[:, 0], svm_demo.support_vectors_[:, 1],
              s=100, facecolors='none', edgecolors='green', linewidth=2)
    
    test_acc = svm_demo.score(X_test, y_test)
    support_ratio = len(svm_demo.support_vectors_) / len(X_train)
    
    ax.set_title(f'C = {C_val}\\nAcc: {test_acc:.3f}, SVs: {len(svm_demo.support_vectors_)}\\nRatio: {support_ratio:.3f}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary of findings
print("\\n🎯 HYPERPARAMETER ANALYSIS SUMMARY:")
print("=" * 50)
print(f"• Optimal C: {grid_search.best_params_['C']}")
print(f"• Optimal Gamma: {grid_search.best_params_['gamma']}")
print(f"• Best Kernel: {grid_search.best_params_['kernel']}")
print(f"• Cross-validation Score: {grid_search.best_score_:.4f}")
print(f"• Test Accuracy: {grid_search.score(X_test, y_test):.4f}")

print("\\n💡 KEY OBSERVATIONS:")
print("=" * 25)
print("• Small C: More regularization, simpler boundary, fewer SVs")
print("• Large C: Less regularization, complex boundary, more SVs")  
print("• Small γ: Wide RBF, smooth boundary")
print("• Large γ: Narrow RBF, complex boundary")
print("• Optimal parameters balance bias-variance tradeoff")
print("• Grid search finds best combination systematically")

In [ ]:
# 🚀 REAL-WORLD APPLICATIONS AND ALGORITHM COMPARISON
print("\n🚀 REAL-WORLD APPLICATIONS AND ALGORITHM COMPARISON")
print("=" * 70)

# Application 1: Image Classification with Handwritten Digits
print("📱 Application 1: Handwritten Digit Recognition")
print("-" * 50)

# Load digits dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# Use subset for faster computation
subset_size = 1000
indices = np.random.choice(len(X_digits), subset_size, replace=False)
X_digits_subset = X_digits[indices]
y_digits_subset = y_digits[indices]

# Convert to binary classification (0-4 vs 5-9)
y_binary_digits = (y_digits_subset >= 5).astype(int)

X_train_digits, X_test_digits, y_train_digits, y_test_digits = train_test_split(
    X_digits_subset, y_binary_digits, test_size=0.3, random_state=42)

# Standardize features
scaler_digits = StandardScaler()
X_train_digits_scaled = scaler_digits.fit_transform(X_train_digits)
X_test_digits_scaled = scaler_digits.transform(X_test_digits)

print(f"Dataset: {X_digits_subset.shape[0]} samples, {X_digits_subset.shape[1]} features")
print(f"Classes: 0-4 vs 5-9 (binary classification)")

# Application 2: Text Classification with News Articles
print("\n📰 Application 2: News Article Classification")
print("-" * 50)

# Load news dataset (subset for faster computation)
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, 
                               shuffle=True, random_state=42)

# Convert to binary classification (religion vs tech/science)
y_news_binary = np.array([1 if cat in ['alt.atheism', 'soc.religion.christian'] else 0 
                         for cat in [categories[i] for i in newsgroups.target]])

# Text preprocessing and feature extraction
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', 
                           min_df=5, max_df=0.7)
X_news = vectorizer.fit_transform(newsgroups.data)

X_train_news, X_test_news, y_train_news, y_test_news = train_test_split(
    X_news, y_news_binary, test_size=0.3, random_state=42)

print(f"Dataset: {X_news.shape[0]} documents, {X_news.shape[1]} TF-IDF features")
print(f"Classes: Religion vs Tech/Science (binary classification)")

# Define algorithms for comparison
algorithms = {
    'SVM (Linear)': SVC(kernel='linear', C=1.0),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

# Function to evaluate algorithms on a dataset
def evaluate_algorithms(X_train, X_test, y_train, y_test, dataset_name):
    """Evaluate multiple algorithms on a dataset"""
    results = {}
    
    print(f"\n🔍 Evaluating algorithms on {dataset_name} dataset:")
    print("-" * 40)
    
    for name, model in algorithms.items():
        start_time = time.time()
        
        # Handle sparse matrices for some algorithms
        if hasattr(X_train, 'toarray'):
            if name in ['K-Nearest Neighbors']:
                X_train_dense = X_train.toarray()
                X_test_dense = X_test.toarray()
                model.fit(X_train_dense, y_train)
                y_pred = model.predict(X_test_dense)
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        train_time = time.time() - start_time
        
        # Compute metrics
        accuracy = accuracy_score(y_test, y_pred)
        
        # Count support vectors for SVM
        n_support_vectors = len(model.support_vectors_) if hasattr(model, 'support_vectors_') else 'N/A'
        
        results[name] = {
            'accuracy': accuracy,
            'train_time': train_time,
            'n_support_vectors': n_support_vectors,
            'model': model
        }
        
        print(f"{name:20s}: Accuracy = {accuracy:.4f}, Time = {train_time:.3f}s, SVs = {n_support_vectors}")
    
    return results

# Evaluate on both applications
print("=" * 70)
digits_results = evaluate_algorithms(X_train_digits_scaled, X_test_digits_scaled, 
                                   y_train_digits, y_test_digits, "Handwritten Digits")

news_results = evaluate_algorithms(X_train_news, X_test_news, 
                                 y_train_news, y_test_news, "News Classification")

# Create comprehensive comparison visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Accuracy comparison for digits
ax1 = axes[0, 0]
digit_names = list(digits_results.keys())
digit_accuracies = [results['accuracy'] for results in digits_results.values()]

bars1 = ax1.bar(digit_names, digit_accuracies, alpha=0.8, color='skyblue')
ax1.set_title('Handwritten Digits Classification\nAccuracy Comparison', fontweight='bold')
ax1.set_ylabel('Accuracy')
ax1.set_xticklabels(digit_names, rotation=45, ha='right')
ax1.set_ylim(0.8, 1.0)
ax1.grid(True, alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars1, digit_accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

# Plot 2: Accuracy comparison for news
ax2 = axes[0, 1]
news_names = list(news_results.keys())
news_accuracies = [results['accuracy'] for results in news_results.values()]

bars2 = ax2.bar(news_names, news_accuracies, alpha=0.8, color='lightcoral')
ax2.set_title('News Classification\nAccuracy Comparison', fontweight='bold')
ax2.set_ylabel('Accuracy')
ax2.set_xticklabels(news_names, rotation=45, ha='right')
ax2.set_ylim(0.8, 1.0)
ax2.grid(True, alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars2, news_accuracies):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
            f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

# Plot 3: Training time comparison
ax3 = axes[0, 2]
digit_times = [results['train_time'] for results in digits_results.values()]
news_times = [results['train_time'] for results in news_results.values()]

x = np.arange(len(digit_names))
width = 0.35

ax3.bar(x - width/2, digit_times, width, label='Digits', alpha=0.8, color='skyblue')
ax3.bar(x + width/2, news_times, width, label='News', alpha=0.8, color='lightcoral')

ax3.set_title('Training Time Comparison', fontweight='bold')
ax3.set_ylabel('Training Time (seconds)')
ax3.set_xlabel('Algorithms')
ax3.set_xticks(x)
ax3.set_xticklabels(digit_names, rotation=45, ha='right')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Support Vector Analysis for Digits
ax4 = axes[1, 0]
svm_digits_linear = digits_results['SVM (Linear)']['model']
svm_digits_rbf = digits_results['SVM (RBF)']['model']

sv_counts = [
    len(svm_digits_linear.support_vectors_),
    len(svm_digits_rbf.support_vectors_)
]
sv_ratios = [
    len(svm_digits_linear.support_vectors_) / len(X_train_digits_scaled),
    len(svm_digits_rbf.support_vectors_) / len(X_train_digits_scaled)
]

svm_types = ['Linear SVM', 'RBF SVM']
ax4_twin = ax4.twinx()

bars4 = ax4.bar(svm_types, sv_counts, alpha=0.8, color='gold')
line4 = ax4_twin.plot(svm_types, sv_ratios, 'ro-', linewidth=2, markersize=8, label='SV Ratio')

ax4.set_title('Support Vector Analysis\n(Digits Dataset)', fontweight='bold')
ax4.set_ylabel('Number of Support Vectors', color='black')
ax4_twin.set_ylabel('Support Vector Ratio', color='red')
ax4_twin.tick_params(axis='y', labelcolor='red')
ax4.grid(True, alpha=0.3)

# Plot 5: Feature importance for Linear SVM (News)
ax5 = axes[1, 1]
linear_svm_news = news_results['SVM (Linear)']['model']
feature_names = vectorizer.get_feature_names_out()

# Get feature weights (coefficients)
coef = linear_svm_news.coef_[0]
top_positive_indices = np.argsort(coef)[-10:][::-1]
top_negative_indices = np.argsort(coef)[:10]

top_features = (list(top_positive_indices) + list(top_negative_indices))
top_weights = coef[top_features]
top_words = [feature_names[i] for i in top_features]

colors = ['green' if w > 0 else 'red' for w in top_weights]
bars5 = ax5.barh(range(len(top_words)), top_weights, color=colors, alpha=0.7)
ax5.set_yticks(range(len(top_words)))
ax5.set_yticklabels(top_words)
ax5.set_title('Most Important Features\n(Linear SVM - News)', fontweight='bold')
ax5.set_xlabel('Feature Weight')
ax5.grid(True, alpha=0.3)

# Plot 6: ROC Curves for best performing models
ax6 = axes[1, 2]

# Get best models
best_digits_model = max(digits_results.items(), key=lambda x: x[1]['accuracy'])[1]['model']
best_news_model = max(news_results.items(), key=lambda x: x[1]['accuracy'])[1]['model']

# Compute ROC for digits
if hasattr(best_digits_model, 'decision_function'):
    y_score_digits = best_digits_model.decision_function(X_test_digits_scaled)
else:
    y_score_digits = best_digits_model.predict_proba(X_test_digits_scaled)[:, 1]

fpr_digits, tpr_digits, _ = roc_curve(y_test_digits, y_score_digits)
auc_digits = roc_auc_score(y_test_digits, y_score_digits)

# Compute ROC for news
if hasattr(best_news_model, 'decision_function'):
    if hasattr(X_test_news, 'toarray'):
        y_score_news = best_news_model.decision_function(X_test_news)
    else:
        y_score_news = best_news_model.decision_function(X_test_news)
else:
    if hasattr(X_test_news, 'toarray'):
        y_score_news = best_news_model.predict_proba(X_test_news.toarray())[:, 1]
    else:
        y_score_news = best_news_model.predict_proba(X_test_news)[:, 1]

fpr_news, tpr_news, _ = roc_curve(y_test_news, y_score_news)
auc_news = roc_auc_score(y_test_news, y_score_news)

# Plot ROC curves
ax6.plot(fpr_digits, tpr_digits, 'b-', linewidth=2, 
         label=f'Digits (AUC = {auc_digits:.3f})')
ax6.plot(fpr_news, tpr_news, 'r-', linewidth=2, 
         label=f'News (AUC = {auc_news:.3f})')
ax6.plot([0, 1], [0, 1], 'k--', alpha=0.5)

ax6.set_xlabel('False Positive Rate')
ax6.set_ylabel('True Positive Rate')
ax6.set_title('ROC Curves\n(Best Performing Models)', fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Create comprehensive results table
print("\n📊 COMPREHENSIVE RESULTS SUMMARY")
print("=" * 80)

comparison_data = []
for app_name, results in [('Digits', digits_results), ('News', news_results)]:
    for alg_name, metrics in results.items():
        comparison_data.append({
            'Application': app_name,
            'Algorithm': alg_name,
            'Accuracy': f"{metrics['accuracy']:.4f}",
            'Training Time (s)': f"{metrics['train_time']:.3f}",
            'Support Vectors': metrics['n_support_vectors']
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print("\n🏆 BEST PERFORMERS:")
print("=" * 20)
best_digits = max(digits_results.items(), key=lambda x: x[1]['accuracy'])
best_news = max(news_results.items(), key=lambda x: x[1]['accuracy'])

print(f"• Handwritten Digits: {best_digits[0]} (Accuracy: {best_digits[1]['accuracy']:.4f})")
print(f"• News Classification: {best_news[0]} (Accuracy: {best_news[1]['accuracy']:.4f})")

print("\n💡 KEY INSIGHTS FROM REAL-WORLD APPLICATIONS:")
print("=" * 50)
print("• SVMs excel in high-dimensional spaces (text classification)")
print("• Linear kernels work well for linearly separable text data")
print("• RBF kernels adapt better to complex image patterns")
print("• Support vector sparsity makes SVMs memory-efficient")
print("• Feature scaling is crucial for SVM performance")
print("• Training time scales with dataset size and complexity")
print("• SVMs provide interpretable feature importance (linear kernel)")

In [ ]:
# 🎮 INTERACTIVE VISUALIZATIONS AND COMPREHENSIVE PERFORMANCE ANALYSIS
print("🎮 INTERACTIVE VISUALIZATIONS AND COMPREHENSIVE PERFORMANCE ANALYSIS")
print("=" * 80)

# 1. Advanced Decision Boundary Visualization with Confidence Regions
print("\n🎨 1. Advanced Decision Boundary Visualization")
print("-" * 55)

def plot_advanced_decision_boundary(X, y, model, title="Advanced SVM Decision Boundary"):
    """
    Create advanced visualization with decision boundary, confidence regions, 
    support vectors, and detailed annotations
    """
    plt.figure(figsize=(14, 10))
    
    # Create fine mesh for smooth boundaries
    h = 0.01
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Get decision function values
    mesh_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.decision_function(mesh_points)
    Z = Z.reshape(xx.shape)
    
    # Plot confidence regions with multiple levels
    confidence_levels = np.linspace(-2, 2, 21)
    contour_filled = plt.contourf(xx, yy, Z, levels=confidence_levels, 
                                 alpha=0.4, cmap='RdYlBu')
    
    # Plot decision boundary and margins with enhanced styling
    boundary_contour = plt.contour(xx, yy, Z, levels=[-1, 0, 1], 
                                  colors=['red', 'black', 'red'],
                                  linestyles=['--', '-', '--'], 
                                  linewidths=[3, 4, 3], alpha=0.8)
    
    # Add labels to contour lines
    plt.clabel(boundary_contour, inline=True, fontsize=12, fmt='%0.0f')
    
    # Plot data points with enhanced styling
    unique_classes = np.unique(y)
    colors = ['red', 'blue'] if len(unique_classes) == 2 else plt.cm.Set1(np.linspace(0, 1, len(unique_classes)))
    
    for i, cls in enumerate(unique_classes):
        mask = (y == cls)
        plt.scatter(X[mask, 0], X[mask, 1], c=colors[i], 
                   s=80, alpha=0.8, edgecolors='black', linewidth=1.5,
                   label=f'Class {cls}', marker='o' if cls == unique_classes[0] else '^')
    
    # Highlight support vectors with special markers
    if hasattr(model, 'support_vectors_'):
        plt.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                   s=200, facecolors='none', edgecolors='lime', 
                   linewidth=3, marker='s', 
                   label=f'Support Vectors ({len(model.support_vectors_)})')
        
        # Add support vector annotations
        for i, sv in enumerate(model.support_vectors_[:min(5, len(model.support_vectors_))]):
            plt.annotate(f'SV{i+1}', (sv[0], sv[1]), 
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=10, color='lime', fontweight='bold')
    
    # Enhanced colorbar
    cbar = plt.colorbar(contour_filled, label='Decision Function Value', shrink=0.8)
    cbar.set_label('Decision Function Value', fontsize=12, fontweight='bold')
    
    # Styling and labels
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Feature 1', fontsize=14, fontweight='bold')
    plt.ylabel('Feature 2', fontsize=14, fontweight='bold')
    plt.legend(loc='upper right', fontsize=12, framealpha=0.9)
    plt.grid(True, alpha=0.3, linestyle=':')
    
    # Add text box with model information
    if hasattr(model, 'C'):
        textstr = f'C = {model.C}\\nKernel = {model.kernel}\\n'
        if hasattr(model, 'gamma'):
            textstr += f'γ = {model.gamma}'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        plt.text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=10,
                verticalalignment='top', bbox=props)
    
    plt.tight_layout()
    return plt.gcf()

# Create advanced visualization for different datasets
datasets_demo = {
    'Circular Pattern': make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42),
    'Moon Pattern': make_moons(n_samples=200, noise=0.15, random_state=42)
}

print("Creating advanced decision boundary visualizations...")
for name, (X, y) in datasets_demo.items():
    # Standardize data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train optimized SVM
    svm_model = SVC(kernel='rbf', C=10, gamma=1.0)
    svm_model.fit(X_scaled, y)
    
    # Create advanced visualization
    plot_advanced_decision_boundary(X_scaled, y, svm_model, 
                                  f'Advanced SVM Visualization: {name}')
    plt.show()

# 2. Comprehensive Performance Dashboard
print("\n📊 2. Comprehensive Performance Dashboard")
print("-" * 50)

def create_performance_dashboard(models_results, dataset_names):
    """Create a comprehensive dashboard showing multiple performance metrics"""
    
    fig = plt.figure(figsize=(20, 16))
    
    # Extract metrics
    accuracies = {}
    training_times = {}
    support_ratios = {}
    
    for dataset in dataset_names:
        accuracies[dataset] = [models_results[dataset][model]['accuracy'] for model in models_results[dataset]]
        training_times[dataset] = [models_results[dataset][model]['train_time'] for model in models_results[dataset]]
        support_ratios[dataset] = [models_results[dataset][model].get('support_ratio', 0) for model in models_results[dataset]]
    
    model_names = list(models_results[dataset_names[0]].keys())
    
    # Plot 1: Accuracy Comparison (Radar Chart)
    ax1 = plt.subplot(2, 3, 1, projection='polar')
    
    angles = np.linspace(0, 2*np.pi, len(model_names), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle
    
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for i, dataset in enumerate(dataset_names):
        values = accuracies[dataset] + [accuracies[dataset][0]]  # Complete the circle
        ax1.plot(angles, values, 'o-', linewidth=2, label=dataset, color=colors[i % len(colors)])
        ax1.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])
    
    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(model_names, fontsize=10)
    ax1.set_ylim(0.7, 1.0)
    ax1.set_title('Accuracy Comparison\\n(Radar Chart)', fontweight='bold', pad=20)
    ax1.legend(loc='upper right', bbox_to_anchor=(1.2, 1.0))
    ax1.grid(True)
    
    # Plot 2: Training Time vs Accuracy
    ax2 = plt.subplot(2, 3, 2)
    
    for i, dataset in enumerate(dataset_names):
        ax2.scatter(training_times[dataset], accuracies[dataset], 
                   s=100, alpha=0.7, label=dataset, color=colors[i % len(colors)])
        
        # Add model labels
        for j, model in enumerate(model_names):
            ax2.annotate(model[:3], (training_times[dataset][j], accuracies[dataset][j]),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax2.set_xlabel('Training Time (seconds)', fontweight='bold')
    ax2.set_ylabel('Accuracy', fontweight='bold')
    ax2.set_title('Efficiency vs Accuracy Trade-off', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Model Complexity Analysis
    ax3 = plt.subplot(2, 3, 3)
    
    x = np.arange(len(model_names))
    width = 0.15
    
    for i, dataset in enumerate(dataset_names[:3]):  # Limit to 3 datasets for clarity
        if any(sr > 0 for sr in support_ratios[dataset]):
            ax3.bar(x + i*width, support_ratios[dataset], width, 
                   label=f'{dataset} - SV Ratio', alpha=0.7, color=colors[i])
    
    ax3.set_xlabel('Models', fontweight='bold')
    ax3.set_ylabel('Support Vector Ratio', fontweight='bold')
    ax3.set_title('Model Complexity\\n(Support Vector Ratio)', fontweight='bold')
    ax3.set_xticks(x + width)
    ax3.set_xticklabels([name[:8] for name in model_names], rotation=45)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Performance Heat Map
    ax4 = plt.subplot(2, 3, 4)
    
    performance_matrix = np.array([accuracies[dataset] for dataset in dataset_names])
    im = ax4.imshow(performance_matrix, cmap='viridis', aspect='auto')
    
    ax4.set_xticks(range(len(model_names)))
    ax4.set_xticklabels([name[:10] for name in model_names], rotation=45)
    ax4.set_yticks(range(len(dataset_names)))
    ax4.set_yticklabels(dataset_names)
    ax4.set_title('Performance Heat Map\\n(Accuracy)', fontweight='bold')
    
    # Add text annotations
    for i in range(len(dataset_names)):
        for j in range(len(model_names)):
            ax4.text(j, i, f'{performance_matrix[i, j]:.3f}', 
                    ha='center', va='center', color='white', fontweight='bold')
    
    plt.colorbar(im, ax=ax4, label='Accuracy')
    
    # Plot 5: Statistical Significance Test
    ax5 = plt.subplot(2, 3, 5)
    
    # Perform paired t-test between SVM models
    svm_linear_scores = [accuracies[dataset][0] for dataset in dataset_names]  # Assuming first is Linear SVM
    svm_rbf_scores = [accuracies[dataset][1] for dataset in dataset_names]     # Assuming second is RBF SVM
    
    from scipy import stats
    t_stat, p_value = stats.ttest_rel(svm_linear_scores, svm_rbf_scores)
    
    # Box plot for comparison
    ax5.boxplot([svm_linear_scores, svm_rbf_scores], 
               labels=['Linear SVM', 'RBF SVM'], patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    
    ax5.set_ylabel('Accuracy', fontweight='bold')
    ax5.set_title(f'Statistical Comparison\\np-value: {p_value:.4f}', fontweight='bold')
    ax5.grid(True, alpha=0.3)
    
    # Add significance annotation
    if p_value < 0.05:
        ax5.text(0.5, 0.95, 'Statistically Significant', transform=ax5.transAxes,
                ha='center', va='top', fontweight='bold', color='red',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
    
    # Plot 6: Learning Curves
    ax6 = plt.subplot(2, 3, 6)
    
    # Generate learning curve data for RBF SVM
    from sklearn.model_selection import learning_curve
    
    # Use one dataset for learning curve
    X_sample, y_sample = datasets_demo['Circular Pattern']
    X_sample_scaled = StandardScaler().fit_transform(X_sample)
    
    train_sizes, train_scores, val_scores = learning_curve(
        SVC(kernel='rbf', C=1.0), X_sample_scaled, y_sample, 
        cv=5, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 10))
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    ax6.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Score')
    ax6.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax6.plot(train_sizes, val_mean, 's-', color='red', label='Validation Score')
    ax6.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    
    ax6.set_xlabel('Training Set Size', fontweight='bold')
    ax6.set_ylabel('Accuracy', fontweight='bold')
    ax6.set_title('Learning Curves\\n(RBF SVM)', fontweight='bold')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Generate performance data for dashboard
dashboard_datasets = ['Circular Pattern', 'Moon Pattern', 'Linear Separable']
models_for_dashboard = {
    'Linear SVM': SVC(kernel='linear', C=1.0),
    'RBF SVM': SVC(kernel='rbf', C=1.0),
    'Poly SVM': SVC(kernel='poly', degree=3, C=1.0)
}

dashboard_results = {}
for dataset_name in dashboard_datasets:
    if dataset_name == 'Linear Separable':
        X_data, y_data = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                                           n_informative=2, n_clusters_per_class=1, 
                                           class_sep=2.0, random_state=42)
    else:
        X_data, y_data = datasets_demo[dataset_name]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_data)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_data, test_size=0.3, random_state=42)
    
    dashboard_results[dataset_name] = {}
    
    for model_name, model in models_for_dashboard.items():
        start_time = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start_time
        
        accuracy = model.score(X_test, y_test)
        support_ratio = len(model.support_vectors_) / len(X_train) if hasattr(model, 'support_vectors_') else 0
        
        dashboard_results[dataset_name][model_name] = {
            'accuracy': accuracy,
            'train_time': train_time,
            'support_ratio': support_ratio
        }

# Create comprehensive dashboard
print("Creating comprehensive performance dashboard...")
dashboard_fig = create_performance_dashboard(dashboard_results, dashboard_datasets)
plt.show()

# 3. Interactive Parameter Exploration
print("\n🎛️ 3. Interactive Parameter Space Exploration")
print("-" * 50)

def explore_parameter_space():
    """Explore SVM parameter space with detailed analysis"""
    
    # Generate test data
    X_explore, y_explore = make_circles(n_samples=300, noise=0.15, factor=0.4, random_state=42)
    scaler = StandardScaler()
    X_explore_scaled = scaler.fit_transform(X_explore)
    
    # Parameter ranges
    C_values = [0.1, 1, 10, 100]
    gamma_values = [0.01, 0.1, 1, 10]
    
    fig, axes = plt.subplots(len(C_values), len(gamma_values), 
                           figsize=(20, 20))
    
    performance_grid = np.zeros((len(C_values), len(gamma_values)))
    
    for i, C in enumerate(C_values):
        for j, gamma in enumerate(gamma_values):
            ax = axes[i, j]
            
            # Train SVM
            svm = SVC(kernel='rbf', C=C, gamma=gamma)
            svm.fit(X_explore_scaled, y_explore)
            
            # Calculate performance
            accuracy = svm.score(X_explore_scaled, y_explore)
            performance_grid[i, j] = accuracy
            
            # Plot decision boundary
            h = 0.02
            x_min, x_max = X_explore_scaled[:, 0].min() - 1, X_explore_scaled[:, 0].max() + 1
            y_min, y_max = X_explore_scaled[:, 1].min() - 1, X_explore_scaled[:, 1].max() + 1
            xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                               np.arange(y_min, y_max, h))
            
            Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
            Z = Z.reshape(xx.shape)
            
            ax.contourf(xx, yy, Z, levels=50, alpha=0.4, cmap='RdYlBu')
            ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['red', 'black', 'red'],
                      linestyles=['--', '-', '--'], linewidths=[2, 3, 2])
            
            # Plot data points
            scatter = ax.scatter(X_explore_scaled[:, 0], X_explore_scaled[:, 1], 
                               c=y_explore, cmap='RdYlBu', s=30, edgecolors='black')
            
            # Highlight support vectors
            ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
                      s=80, facecolors='none', edgecolors='green', linewidth=2)
            
            # Title with parameters and metrics
            n_sv = len(svm.support_vectors_)
            sv_ratio = n_sv / len(X_explore_scaled)
            ax.set_title(f'C={C}, γ={gamma}\\nAcc:{accuracy:.3f}, SVs:{n_sv}\\nRatio:{sv_ratio:.3f}',
                        fontsize=10, fontweight='bold')
            ax.set_xticks([])
            ax.set_yticks([])
    
    # Add row and column labels
    for i, C in enumerate(C_values):
        axes[i, 0].set_ylabel(f'C = {C}', fontsize=12, fontweight='bold')
    
    for j, gamma in enumerate(gamma_values):
        axes[0, j].set_xlabel(f'γ = {gamma}', fontsize=12, fontweight='bold')
        axes[0, j].xaxis.set_label_position('top')
    
    plt.suptitle('SVM Parameter Space Exploration: C vs Gamma', 
                fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    return performance_grid

print("Exploring parameter space...")
param_performance = explore_parameter_space()

# 4. Final Summary and Recommendations
print("\n🎯 4. FINAL PERFORMANCE ANALYSIS & RECOMMENDATIONS")
print("=" * 70)

# Create summary statistics
summary_stats = {
    'Best Overall Accuracy': 0,
    'Most Efficient Model': '',
    'Most Complex Model': '',
    'Most Robust Model': ''
}

# Find best performing configurations
all_accuracies = []
all_models = []
all_datasets = []

for dataset_name, results in dashboard_results.items():
    for model_name, metrics in results.items():
        all_accuracies.append(metrics['accuracy'])
        all_models.append(model_name)
        all_datasets.append(dataset_name)

best_idx = np.argmax(all_accuracies)
summary_stats['Best Overall Accuracy'] = all_accuracies[best_idx]
best_combination = f"{all_models[best_idx]} on {all_datasets[best_idx]}"

print(f"🏆 PERFORMANCE SUMMARY:")
print(f"   • Best Accuracy: {summary_stats['Best Overall Accuracy']:.4f} ({best_combination})")
print(f"   • Average Accuracy across all configurations: {np.mean(all_accuracies):.4f}")
print(f"   • Standard deviation: {np.std(all_accuracies):.4f}")

# Parameter space analysis
best_param_idx = np.unravel_index(np.argmax(param_performance), param_performance.shape)
C_values = [0.1, 1, 10, 100]
gamma_values = [0.01, 0.1, 1, 10]
optimal_C = C_values[best_param_idx[0]]
optimal_gamma = gamma_values[best_param_idx[1]]

print(f"\\n🔧 OPTIMAL PARAMETERS:")
print(f"   • Best C: {optimal_C}")
print(f"   • Best Gamma: {optimal_gamma}")
print(f"   • Best Parameter Accuracy: {param_performance[best_param_idx]:.4f}")

print("\\n💡 COMPREHENSIVE INSIGHTS & RECOMMENDATIONS:")
print("=" * 55)
print("✅ KERNEL SELECTION GUIDELINES:")
print("   • Linear: Use for high-dimensional, linearly separable data")
print("   • RBF: Default choice for most non-linear problems")
print("   • Polynomial: Specific polynomial relationships in data")
print("   • Sigmoid: Neural network-like behavior (use cautiously)")

print("\\n✅ HYPERPARAMETER TUNING RECOMMENDATIONS:")
print("   • Start with C=1.0 and gamma='scale' for RBF kernel")
print("   • Use grid search or random search for optimization")  
print("   • Small C → more regularization (simpler model)")
print("   • Large C → less regularization (complex model)")
print("   • Small γ → wide RBF influence (smoother boundaries)")
print("   • Large γ → narrow RBF influence (complex boundaries)")

print("\\n✅ PRACTICAL APPLICATION GUIDELINES:")
print("   • Always standardize features before training")
print("   • Use cross-validation for reliable performance estimates")
print("   • Monitor support vector ratio for complexity indication")
print("   • Consider training time vs accuracy trade-offs")
print("   • Linear SVM provides interpretable feature importance")

print("\\n✅ WHEN TO USE SVM:")
print("   • High-dimensional data (text, genomics)")
print("   • Clear margin of separation exists")
print("   • Memory efficiency is important (sparse solution)")
print("   • Strong theoretical foundation is preferred")

print("\\n✅ WHEN TO AVOID SVM:")
print("   • Very large datasets (>100K samples)")
print("   • Probabilistic output is required")  
print("   • Features are not normalized/standardized")
print("   • Interpretability is critical (except linear SVM)")

print("\\n🎓 EDUCATIONAL OBJECTIVES ACHIEVED:")
print("=" * 40)
print("✓ Complete mathematical understanding of SVM theory")
print("✓ Implementation of SMO algorithm from scratch") 
print("✓ Comprehensive kernel analysis and comparison")
print("✓ Hyperparameter optimization techniques")
print("✓ Real-world applications and performance evaluation")
print("✓ Interactive visualizations and decision boundaries")
print("✓ Statistical analysis and model comparison")
print("✓ Practical guidelines and best practices")

print("\\n🚀 NEXT STEPS FOR ADVANCED LEARNING:")
print("=" * 40)
print("• Study SVM extensions: Multi-class, regression (SVR)")
print("• Explore kernel design and custom kernel functions")
print("• Investigate online SVM algorithms for streaming data")
print("• Learn about structured SVMs for sequence labeling")
print("• Apply SVMs to domain-specific problems")
print("• Compare with deep learning approaches")

print("\\n" + "="*80)
print("🎉 SVM NOTEBOOK ENHANCEMENT COMPLETED SUCCESSFULLY! 🎉")
print("="*80)